In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-mpf qiskit-addon-utils scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Multi-Produkt-Formeln zur Reduzierung des Trotter-Fehlers

import TutorialFeedback from '@site/src/components/TutorialFeedback';




*Geschätzte QPU-Nutzung: Vier Minuten auf einem Heron-r2-Prozessor (HINWEIS: Dies ist nur eine Schätzung. Deine tatsächliche Laufzeit kann abweichen.)*

## Hintergrund
Dieses Tutorial zeigt, wie du eine Multi-Produkt-Formel (MPF) einsetzt, um einen geringeren Trotter-Fehler beim Observable zu erzielen, als er durch den tiefsten Trotter-Schaltkreis entsteht, den du tatsächlich ausführst.
MPFs reduzieren den Trotter-Fehler der Hamiltonschen Dynamik durch eine gewichtete Kombination mehrerer Schaltkreisausführungen. Betrachte die Aufgabe, Erwartungswerte von Observablen für den Quantenzustand
$\rho(t)=e^{-i H t} \rho(0) e^{i H t}$ mit dem Hamiltonian $H$ zu berechnen. Man kann Produkt-Formeln (PFs) verwenden, um die Zeitentwicklung $e^{-i H t}$ wie folgt anzunähern:

- Schreibe den Hamiltonian $H$ als $H=\sum_{a=1}^d F_a,$ wobei $F_a$ hermitesche Operatoren sind, sodass jedes zugehörige Unitär effizient auf einem Quantenprozessor implementiert werden kann.
- Nähere die Terme $F_a$ an, die nicht miteinander kommutieren.

Die Produkt-Formel erster Ordnung (Lie-Trotter-Formel) lautet dann:

$$S_1(t):=\prod_{a=1}^d e^{-i F_a t},$$

mit dem quadratischen Fehlerterm $S_1(t)=e^{-i H t}+\mathcal{O}\left(t^{2}\right)$. Es gibt auch Produkt-Formeln höherer Ordnung (Lie-Trotter-Suzuki-Formeln), die schneller konvergieren und rekursiv definiert sind:

$$S_2(t):=\prod_{a=1}^d e^{-i F_a t/2}\prod_{a=1}^d e^{-i F_a t/2}$$

$$S_{2 \chi}(t):= S_{2 \chi -2}(s_{\chi}t)^2 S_{2 \chi -2}((1-4s_{\chi})t)S_{2 \chi -2}(s_{\chi}t)^2,$$

wobei $\chi$ die Ordnung der symmetrischen PF und $s_p = \left( 4 - 4^{1/(2p-1)} \right)^{-1}$ ist. Bei langen Zeitentwicklungen kann man das Zeitintervall $t$ in $k$ Abschnitte – sogenannte Trotter-Schritte – der Länge $t/k$ unterteilen und die Zeitentwicklung in jedem Abschnitt mit einer Produkt-Formel der Ordnung $\chi$, $S_{\chi}$, annähern. Die PF der Ordnung $\chi$ über $k$ Trotter-Schritte lautet damit:

$$ S_{\chi}^{k}(t) = \left[ S_{\chi} \left( \frac{t}{k} \right)\right]^k = e^{-i H t}+O\left(t \left( \frac{t}{k} \right)^{\chi} \right)$$

wobei der Fehlerterm mit der Anzahl der Trotter-Schritte $k$ und der Ordnung $\chi$ der PF abnimmt.

Für eine ganze Zahl $k \geq 1$ und eine Produkt-Formel $S_{\chi}(t)$ erhält man den näherungsweise zeitentwickelten Zustand $\rho_k(t)$ aus $\rho_0$, indem man $k$ Iterationen der Produkt-Formel $S_{\chi}\left(\frac{t}{k}\right)$ anwendet.

$$
\rho_k(t)=S_{\chi}\left(\frac{t}{k}\right)^k \rho_0 S_{\chi}\left(\frac{t}{k}\right)^{-k}
$$

$\rho_k(t)$ ist eine Näherung für $\rho(t)$ mit dem Trotter-Approximationsfehler ||$\rho_k(t)-\rho(t) ||$. Betrachten wir eine Linearkombination von Trotter-Näherungen von $\rho(t)$:

$$
\mu(t) = \sum_{j}^{l} x_j \rho^{k_j}_{j}\left(\frac{t}{k_j}\right) + \text{verbleibender Trotter-Fehler} \, ,
$$

wobei $x_j$ die Gewichtungskoeffizienten sind, $\rho^{k_j}_j$ die Dichtematrix des reinen Zustands ist, der durch Entwicklung des Anfangszustands mit der Produkt-Formel $S^{k_j}_{\chi}$ mit $k_j$ Trotter-Schritten entsteht, und $j \in {1, ..., l}$ die Anzahl der PFs indiziert, die die MPF bilden. Alle Terme in $\mu(t)$ verwenden dieselbe Produkt-Formel $S_{\chi}(t)$ als Basis.
Das Ziel ist es, ||$\rho_k(t)-\rho(t) \|$ zu verbessern, indem man ein $\mu(t)$ mit noch kleinerem $\|\mu(t)-\rho(t)\|$ findet.

* $\mu(t)$ muss kein physikalischer Zustand sein, da $x_i$ nicht positiv sein müssen. Ziel ist es, den Fehler im Erwartungswert der Observablen zu minimieren, nicht einen physikalischen Ersatz für $\rho(t)$ zu finden.
* $k_j$ bestimmt sowohl die Schaltkreistiefe als auch das Niveau der Trotter-Näherung. Kleinere Werte von $k_j$ führen zu flacheren Schaltkreisen, die weniger Schaltkreisfehler verursachen, aber eine weniger genaue Näherung des gewünschten Zustands darstellen.

Der entscheidende Punkt ist, dass der verbleibende Trotter-Fehler von $\mu(t)$ kleiner ist als der Trotter-Fehler, den man durch einfache Verwendung des größten $k_j$-Werts erhalten würde.

Dieser Ansatz lässt sich aus zwei Perspektiven betrachten:

1. Bei einem festen Budget an Trotter-Schritten kannst du Ergebnisse mit insgesamt kleinerem Trotter-Fehler erzielen.
2. Wenn eine Zielanzahl von Trotter-Schritten zu groß ist, um sie auszuführen, kannst du die MPF nutzen, um eine Sammlung von Schaltkreisen mit geringerer Tiefe zu finden, die einen ähnlichen Trotter-Fehler ergeben.
## Voraussetzungen
Stelle vor Beginn dieses Tutorials sicher, dass Folgendes installiert ist:

* Qiskit SDK v1.0 oder höher, mit Unterstützung für [Visualisierung](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime v0.22 oder höher (`pip install qiskit-ibm-runtime`)
* MPF-Qiskit-Addons (`pip install qiskit_addon_mpf`)
* Qiskit-Addons-Utils (`pip install qiskit_addon_utils`)
* Quimb-Bibliothek (`pip install quimb`)
* Qiskit-Quimb-Bibliothek (`pip install qiskit-quimb`)
* Numpy v0.21 für paketübergreifende Kompatibilität (`pip install numpy==0.21`)
## Teil I. Kleinskaliges Beispiel
### Stabilität der MPF untersuchen
Es gibt keine offensichtliche Einschränkung bei der Wahl der Anzahl der Trotter-Schritte $k_j$, die den MPF-Zustand $\mu(t)$ bilden. Diese müssen jedoch sorgfältig gewählt werden, um Instabilitäten in den aus $\mu(t)$ berechneten Erwartungswerten zu vermeiden. Eine gute Faustregel ist, den kleinsten Trotter-Schritt $k_{\text{min}}$ so zu setzen, dass $t/k_{\text{min}} \lt 1$ gilt. Wenn du mehr darüber erfahren möchtest und wissen willst, wie du deine anderen $k_j$-Werte wählst, lies die Anleitung [How to choose the Trotter steps for an MPF](https://qiskit.github.io/qiskit-addon-mpf/how_tos/choose_trotter_steps.html).

Im folgenden Beispiel untersuchen wir die Stabilität der MPF-Lösung, indem wir den Erwartungswert der Magnetisierung für einen Bereich von Zeiten mit verschiedenen zeitentwickelten Zuständen berechnen. Konkret vergleichen wir die Erwartungswerte der einzelnen näherungsweisen Zeitentwicklungen mit den entsprechenden Trotter-Schritten und der verschiedenen MPF-Modelle (statische und dynamische Koeffizienten) mit den exakten Werten des zeitentwickelten Observablen. Zunächst definieren wir die Parameter für die Trotter-Formeln und die Entwicklungszeiten:

In [1]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from functools import partial
from copy import deepcopy

from qiskit import QuantumCircuit
from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector
from qiskit.synthesis import SuzukiTrotter
from qiskit.transpiler import CouplingMap, PassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.library import XXPlusYYGate
from qiskit.transpiler.passes.optimization.collect_and_collapse import (
    CollectAndCollapse,
    collect_using_filter_function,
    collapse_to_operation,
)

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

from qiskit_addon_utils.problem_generators import (
    generate_xyz_hamiltonian,
    generate_time_evolution_circuit,
)
from qiskit_addon_utils.slicing import slice_by_depth
from qiskit_addon_mpf.static import setup_static_lse
from qiskit_addon_mpf.dynamic import setup_dynamic_lse
from qiskit_addon_mpf.costs import (
    setup_exact_problem,
    setup_sum_of_squares_problem,
    setup_frobenius_problem,
)
from qiskit_addon_mpf.backends.tenpy_layers import (
    LayerModel,
    LayerwiseEvolver,
)
from qiskit_addon_mpf.backends.tenpy_tebd import MPOState, MPS_neel_state

from scipy.linalg import expm

# Suppress TeNPy's `unit_cell_width` future-API warning. The default
# (`unit_cell_width=len(sites)`) is correct for Chain lattices, which is what
# `CouplingMap.from_line(...)` produces here, so the warning is informational.
warnings.filterwarnings(
    "ignore",
    message=r".*unit_cell_width.*",
    category=UserWarning,
)


# --- Helper: collect XX + YY rotations into a single gate ---
def filter_function(node):
    return node.op.name in {"rxx", "ryy"}


collect_function = partial(
    collect_using_filter_function,
    filter_function=filter_function,
    split_blocks=True,
    min_block_size=1,
)


def collapse_to_xx_plus_yy(block):
    param = 0.0
    for node in block.data:
        param += node.operation.params[0]
    return XXPlusYYGate(param)


collapse_function = partial(
    collapse_to_operation,
    collapse_function=collapse_to_xx_plus_yy,
)

pm = PassManager()
pm.append(CollectAndCollapse(collect_function, collapse_function))

Für dieses Beispiel verwenden wir den Néel-Zustand als Anfangszustand $\vert \text{Neel} \rangle = \vert 0101...01 \rangle$ und das Heisenberg-Modell auf einer Kette von 10 Gitterplätzen als Hamilton-Operator für die Zeitentwicklung:

$$
\hat{\mathcal{H}}_{Heis} = J \sum_{i=1}^{L-1} \left(X_i X_{(i+1)}+Y_i Y_{(i+1)}+ Z_i Z_{(i+1)} \right) \, ,
$$

wobei $J$ die Kopplungsstärke für nächste-Nachbar-Kanten ist.

In [2]:
L = 10

# Generate coupling map and Hamiltonian
coupling_map = CouplingMap.from_line(L, bidirectional=False)

hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(1.0, 1.0, 1.0),
    ext_magnetic_field=(0.0, 0.0, 0.0),
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIXXI', 'IIIIIIIYYI', 'IIIIIIIZZI', 'IIIIIXXIII', 'IIIIIYYIII', 'IIIIIZZIII', 'IIIXXIIIII', 'IIIYYIIIII', 'IIIZZIIIII', 'IXXIIIIIII', 'IYYIIIIIII', 'IZZIIIIIII', 'IIIIIIIIXX', 'IIIIIIIIYY', 'IIIIIIIIZZ', 'IIIIIIXXII', 'IIIIIIYYII', 'IIIIIIZZII', 'IIIIXXIIII', 'IIIIYYIIII', 'IIIIZZIIII', 'IIXXIIIIII', 'IIYYIIIIII', 'IIZZIIIIII', 'XXIIIIIIII', 'YYIIIIIIII', 'ZZIIIIIIII'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])


In [3]:
# Observable: ZZ on the middle pair of qubits
observable = SparsePauliOp.from_sparse_list(
    [("ZZ", (L // 2 - 1, L // 2), 1.0)], num_qubits=L
)
print(observable)

SparsePauliOp(['IIIIZZIIII'],
              coeffs=[1.+0.j])


In [4]:
# MPF parameters
mpf_trotter_steps = [1, 2, 4]
order = 2
symmetric = False

trotter_times = np.arange(0.5, 1.55, 0.1)
exact_evolution_times = np.arange(trotter_times[0], 1.55, 0.05)

#### Build Trotter circuits

We create the circuits implementing the approximate Trotter time-evolutions for each time point and each Trotter step count. The `CollectAndCollapse` pass defined in the Setup section collects XX and YY rotations into single XX+YY gates, to prepare for more efficient tensor-network simulation later.

In [5]:
# Initial Neel state preparation
initial_state_circ = QuantumCircuit(L)
initial_state_circ.x([i for i in range(L) if i % 2 != 0])


all_circs = []
for total_time in trotter_times:
    mpf_trotter_circs = [
        generate_time_evolution_circuit(
            hamiltonian,
            time=total_time,
            synthesis=SuzukiTrotter(reps=num_steps, order=order),
        )
        for num_steps in mpf_trotter_steps
    ]

    mpf_trotter_circs = pm.run(
        mpf_trotter_circs
    )  # Collect XX and YY into XX + YY

    mpf_circuits = [
        initial_state_circ.compose(circuit) for circuit in mpf_trotter_circs
    ]
    all_circs.append(mpf_circuits)

In [6]:
mpf_circuits[-1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/multi-product-formula/extracted-outputs/c7ee61e7-0.avif" alt="Output of the previous code cell" />

Anschließend erstellen wir die Schaltkreise, die die näherungsweisen Trotter-Zeitentwicklungen implementieren.

In [7]:
aer_sim = AerSimulator()
pm_sim = generate_preset_pass_manager(backend=aer_sim, optimization_level=3)

isa_circs_all_times = [
    pm_sim.run([deepcopy(c) for c in mpf_circuits])
    for mpf_circuits in all_circs
]

### Step 3: Execute using Qiskit primitives

For the small-scale example we run the ISA-lowered Trotter circuits through the `EstimatorV2` primitive backed by Aer. Doing so gives us a *noiseless* reference value for each $(k_j, t)$ pair &mdash; these are the $\langle A \rangle_{k_j}(t)$ values that the MPF will combine in Step 4. We sweep over evolution times so that we can later plot the full time-series curve of each individual product formula and of the MPF.

In [8]:
estimator = Estimator(mode=aer_sim)

mpf_expvals_all_times, mpf_stds_all_times = [], []
for isa_circuits in isa_circs_all_times:
    result = estimator.run(
        [(circuit, observable) for circuit in isa_circuits], precision=0.005
    ).result()
    mpf_expvals_all_times.append([res.data.evs for res in result])
    mpf_stds_all_times.append([res.data.stds for res in result])

![Output of the previous code cell](../docs/images/tutorials/multi-product-formula/extracted-outputs/92dc20a7-0.avif)

Als nächstes berechnen wir die zeitentwickelten Erwartungswerte aus den Trotter-Schaltkreisen.

In [9]:
exact_expvals = []
for t in exact_evolution_times:
    exp_H = expm(-1j * t * hamiltonian.to_matrix())
    initial_state = Statevector(initial_state_circ).data
    time_evolved_state = exp_H @ initial_state

    exact_obs = (
        time_evolved_state.conj()
        @ observable.to_matrix()
        @ time_evolved_state
    ).real
    exact_expvals.append(exact_obs)

Wir berechnen außerdem die exakten Erwartungswerte zum Vergleich.

In [10]:
lse = setup_static_lse(mpf_trotter_steps, order=order, symmetric=symmetric)

#### Statische MPF-Koeffizienten
Statische MPFs sind solche, bei denen die $x_j$-Werte nicht von der Entwicklungszeit $t$ abhängen. Betrachten wir die PF der Ordnung $\chi = 1$ mit $k_j$ Trotter-Schritten, die sich wie folgt schreiben lässt:

$$ S_1^{k_j}\left( \frac{t}{k_j} \right)=e^{-i H t}+ \sum_{n=1}^{\infty} A_n \frac{t^{n+1}}{k_j^n}  $$

wobei $A_n$ Matrizen sind, die von den Kommutatoren der $F_a$-Terme in der Zerlegung des Hamiltonians abhängen. Wichtig ist, dass $A_n$ selbst unabhängig von der Zeit und der Anzahl der Trotter-Schritte $k_j$ sind. Daher lassen sich Fehlerterme niedrigerer Ordnung in $\mu(t)$ durch eine sorgfältige Wahl der Gewichte $x_j$ der Linearkombination aufheben. Um den Trotter-Fehler der ersten $l-1$ Terme (die den größten Beitrag liefern, da sie einer kleineren Anzahl von Trotter-Schritten entsprechen) im Ausdruck für $\mu(t)$ zu eliminieren, müssen die Koeffizienten $x_j$ folgende Gleichungen erfüllen:

$$ \sum_{j=1}^l x_j = 1 $$
$$ \sum_{j=1}^{l-1} \frac{x_j}{k_j^{n}} = 0 $$

mit $n=0, ... l-2$. Die erste Gleichung stellt sicher, dass der konstruierte Zustand $\mu(t)$ keine Verzerrung aufweist, während die zweite Gleichung die Aufhebung der Trotter-Fehler garantiert. Für Produkt-Formeln höherer Ordnung wird die zweite Gleichung zu $ \sum_{j=1}^{l-1} \frac{x_j}{k_j^{\eta}} = 0 $, wobei $\eta = \chi + 2n$ für symmetrische PFs und $\eta = \chi + n$ sonst gilt, mit $n=0, ..., l-2$. Der resultierende Fehler (Refs. [\[1\]](#references),[\[2\]](#references)) beträgt dann:

$$ \epsilon = \mathcal{O} \left( \frac{t^{l+1}}{k_1^l} \right).$$

Die statischen MPF-Koeffizienten für eine gegebene Menge von $k_j$-Werten zu bestimmen, bedeutet, das lineare Gleichungssystem aus den beiden obigen Gleichungen für die Variablen $x_j$ zu lösen: $Ax=b$. Dabei sind $x$ die gesuchten Koeffizienten, $A$ eine Matrix, die von $k_j$ und der Art der verwendeten PF ($S$) abhängt, und $b$ ein Vektor von Randbedingungen. Konkret gilt:

$$A_{0,j} = 1 $$
$$A_{i>0,j} = k_{j}^{-(\chi + s(i-1))}$$
$$b_0 = 1$$
$$b_{i>0} = 0 $$

wobei $\chi$ die ``order`` ist, $s$ gleich $2$ ist, wenn ``symmetric`` gleich ``True`` ist, und sonst $1$, $k_{j}$ die ``trotter_steps`` sind und $x$ die zu lösenden Variablen sind. Die Indizes $i$ und $j$ beginnen bei $0$. In Matrixform lässt sich dies so darstellen:

$$
A =
\begin{bmatrix}
A_{0,0} & A_{0,1} & A_{0,2} & ... \\
A_{1,0} & A_{1,1} & A_{1,2} & ...  \\
A_{2,0} & A_{2,1} & A_{2,2} & ...  \\
... & ... & ... & ...
\end{bmatrix} =
\begin{bmatrix}
1 & 1 & 1 & ... \\
k_{0}^{-(\chi + s(1-1))} & k_{1}^{-(\chi + s(1-1))} & k_{2}^{-(\chi + s(1-1))} & ... \\
k_{0}^{-(\chi + s(2-1))} & k_{1}^{-(\chi + s(2-1))} & k_{2}^{-(\chi + s(2-1))} & ... \\
... & ... & ... & ...
\end{bmatrix}
$$

und

$$
b =
\begin{bmatrix}
b_{0} \\
b_{1} \\
b_{2}  \\
...
\end{bmatrix} =
\begin{bmatrix}
1 \\
0 \\
0  \\
...
\end{bmatrix}
$$

Weitere Details findest du in der Dokumentation des Linearen Gleichungssystems ([LSE](https://qiskit.github.io/qiskit-addon-mpf/stubs/qiskit_addon_mpf.static.LSE.html)).

Die analytische Lösung für $x$ lautet $x = A^{-1}b$; siehe zum Beispiel Refs. [\[1\]](#references) oder [\[2\]](#references).
Diese exakte Lösung kann jedoch „schlecht konditioniert" sein, was zu sehr großen L1-Normen unserer Koeffizienten $x$ führt und die MPF-Leistung verschlechtern kann.
Alternativ kann man eine Näherungslösung berechnen, die die L1-Norm von $x$ minimiert, um das MPF-Verhalten zu optimieren.
##### Das LGS aufstellen
Nachdem wir unsere $k_j$-Werte gewählt haben, müssen wir zunächst das LGS $Ax=b$ wie oben beschrieben aufstellen.
Die Matrix $A$ hängt nicht nur von $k_j$ ab, sondern auch von unserer Wahl der PF, insbesondere ihrer _Ordnung_.
Zusätzlich kann man berücksichtigen, ob die PF symmetrisch ist oder nicht (siehe [\[1\]](#references)), indem man `symmetric=True/False` setzt.
Dies ist jedoch nicht erforderlich, wie Ref. [\[2\]](#references) zeigt.

In [11]:
lse.A

array([[1.      , 1.      , 1.      ],
       [1.      , 0.25    , 0.0625  ],
       [1.      , 0.125   , 0.015625]])

In [12]:
lse.b

array([1., 0., 0.])

With the LSE in hand, we solve for the static coefficients $x_j$ via `lse.solve()` (this is the direct $x = A^{-1}b$ solution).

In [13]:
mpf_coeffs = lse.solve()
print(
    f"The static coefficients associated with the ansatze are: {mpf_coeffs}"
)

The static coefficients associated with the ansatze are: [ 0.04761905 -0.57142857  1.52380952]


Der Constraint-Vektor $b$ hat folgende Elemente:
$$ b_{0} = 1 $$
$$ b_1 = b_2 = 0 $$

Also:

$$
b =
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

Und entsprechend in `lse`:

In [14]:
model_exact, coeffs_exact = setup_exact_problem(lse)
model_exact.solve()
print(coeffs_exact.value)

[ 0.04761905 -0.57142857  1.52380952]


In [15]:
print(
    "L1 norm of the exact coefficients:",
    np.linalg.norm(coeffs_exact.value, ord=1),
)

L1 norm of the exact coefficients: 2.1428571428556378


Das `lse`-Objekt besitzt Methoden zur Berechnung der statischen Koeffizienten $x_j$, die das Gleichungssystem erfüllen.

In [16]:
model_approx, coeffs_approx = setup_sum_of_squares_problem(
    lse, max_l1_norm=1.5
)
model_approx.solve()
print(coeffs_approx.value)
print(
    "L1 norm of the approximate coefficients:",
    np.linalg.norm(coeffs_approx.value, ord=1),
)

[-1.10294118e-03 -2.48897059e-01  1.25000000e+00]
L1 norm of the approximate coefficients: 1.5


#### Dynamic MPF coefficients

The static MPF cancels Trotter-error terms in a Hamiltonian- and state-agnostic way, so it does not necessarily produce the smallest possible approximation error for a given Hamiltonian and initial state. The dynamic MPF (Refs. [\[2\]](#references), [\[3\]](#references)) instead finds time-dependent coefficients $x_i(t)$ that minimize the Frobenius-norm distance $\|\rho(t) - \mu^D(t)\|_F^2$ at each time $t$. As shown in the Background, this requires the overlap matrix $M_{ij}(t)$ between Trotter-evolved states and the overlap $L_i(t)$ with the exact state &mdash; both of which we estimate using tensor-network (TeNPy) backends in `qiskit_addon_mpf`.

To set up the dynamic LSE we need three ingredients:

1. An **approximate evolver factory** that the addon will run for each $k_j$ to produce $\rho_{k_j}(t)$ as an MPS/MPO. We build it from the layered structure of the order-$2$ Trotter circuit (one layer per `slice_by_depth`), wrapped as a `LayerwiseEvolver` with TeNPy truncation parameters.
2. An **exact evolver factory** that produces a high-accuracy reference $\rho(t)$. We use a small-time-step fourth-order Suzuki-Trotter circuit (`dt=0.1`, `order=4`) as a proxy for exact evolution.
3. An **identity factory** and an **initial-state MPS** that seed the TeNPy simulation.

The cell below constructs the approximate evolver factory.

In [17]:
# Create approximate time-evolution circuits
single_2nd_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=order)
)
single_2nd_order_circ = pm.run(single_2nd_order_circ)  # collect XX and YY

# Find layers in the circuit
layers = slice_by_depth(single_2nd_order_circ, max_slice_depth=1)

# Create tensor network models
models = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz") for layer in layers
]

# Create the time-evolution object
approx_factory = partial(
    LayerwiseEvolver,
    layers=models,
    options={
        "preserve_norm": False,
        "trunc_params": {
            "chi_max": 64,
            "svd_min": 1e-8,
            "trunc_cut": None,
        },
        "max_delta_t": 2,
    },
)

##### $x$ mit einem exakten Modell optimieren
Alternativ zur Berechnung von $x=A^{-1}b$ kannst du [setup_exact_model](https://qiskit.github.io/qiskit-addon-mpf/stubs/qiskit_addon_mpf.static.setup_exact_model.html) verwenden, um eine [cvxpy.Problem](https://www.cvxpy.org/api_reference/cvxpy.problems.html#cvxpy.Problem)-Instanz zu konstruieren, die das LGS als Nebenbedingungen verwendet und deren optimale Lösung $x$ liefert.

Im nächsten Abschnitt wird deutlich, warum diese Schnittstelle existiert.

In [18]:
single_4th_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=4)
)
single_4th_order_circ = pm.run(single_4th_order_circ)
exact_model_layers = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz")
    for layer in slice_by_depth(single_4th_order_circ, max_slice_depth=1)
]

exact_factory = partial(
    LayerwiseEvolver,
    layers=exact_model_layers,
    dt=0.1,
    options={
        "preserve_norm": False,
        "trunc_params": {
            "chi_max": 64,
            "svd_min": 1e-8,
            "trunc_cut": None,
        },
        "max_delta_t": 2,
    },
)

Finally, we define an `identity_factory` that yields the initial MPO state and prepare the Néel initial state as an MPS that matches the lattice used by the layered Trotter model.

In [19]:
def identity_factory():
    return MPOState.initialize_from_lattice(models[0].lat, conserve=True)


mps_initial_state = MPS_neel_state(models[0].lat)

Als Indikator dafür, ob eine MPF mit diesen Koeffizienten gute Ergebnisse liefert, kann die L1-Norm verwendet werden (siehe auch Ref. [\[1\]](#references)).

In [20]:
mpf_dynamic_coeffs_list = []
for t in trotter_times:
    print(f"Computing dynamic coefficients for time={t}")
    lse = setup_dynamic_lse(
        mpf_trotter_steps,
        t,
        identity_factory,
        exact_factory,
        approx_factory,
        mps_initial_state,
    )
    problem, coeffs = setup_frobenius_problem(lse)
    try:
        problem.solve()
        mpf_dynamic_coeffs_list.append(coeffs.value)
    except Exception as error:
        mpf_dynamic_coeffs_list.append(np.zeros(len(mpf_trotter_steps)))
        print(error, "Calculation Failed for time", t)
    print("")

Computing dynamic coefficients for time=0.5

Computing dynamic coefficients for time=0.6

Computing dynamic coefficients for time=0.7

Computing dynamic coefficients for time=0.7999999999999999

Computing dynamic coefficients for time=0.8999999999999999

Computing dynamic coefficients for time=0.9999999999999999

Computing dynamic coefficients for time=1.0999999999999999

Computing dynamic coefficients for time=1.1999999999999997

Computing dynamic coefficients for time=1.2999999999999998

Computing dynamic coefficients for time=1.4

Computing dynamic coefficients for time=1.4999999999999998



#### Combine Trotter expectation values with the MPF coefficients

Now we evaluate $\langle A \rangle_{\text{MPF}}(t) = \sum_j x_j \, \langle A \rangle_{k_j}(t)$ for each set of coefficients (static-exact, static-approximate, and dynamic), propagate the per-circuit standard errors, and plot the resulting time series against the exact-diagonalization curve.

In [21]:
sym = {1: "^", 2: "s", 4: "p"}
# Get expectation values at all times for each Trotter step
for k, step in enumerate(mpf_trotter_steps):
    trotter_curve, trotter_curve_error = [], []
    for trotter_expvals, trotter_stds in zip(
        mpf_expvals_all_times, mpf_stds_all_times
    ):
        trotter_curve.append(trotter_expvals[k])
        trotter_curve_error.append(trotter_stds[k])

    plt.errorbar(
        trotter_times,
        trotter_curve,
        yerr=trotter_curve_error,
        alpha=0.5,
        markersize=4,
        marker=sym[step],
        color="grey",
        label=f"{mpf_trotter_steps[k]} Trotter steps",
    )

# Get expectation values at all times for the static MPF with exact coeffs
exact_mpf_curve, exact_mpf_curve_error = [], []
for trotter_expvals, trotter_stds in zip(
    mpf_expvals_all_times, mpf_stds_all_times
):
    mpf_std = np.sqrt(
        sum(
            [
                (coeff**2) * (std**2)
                for coeff, std in zip(coeffs_exact.value, trotter_stds)
            ]
        )
    )
    exact_mpf_curve_error.append(mpf_std)
    exact_mpf_curve.append(trotter_expvals @ coeffs_exact.value)

plt.errorbar(
    trotter_times,
    exact_mpf_curve,
    yerr=exact_mpf_curve_error,
    markersize=4,
    marker="o",
    label="Static MPF - Exact",
    color="purple",
)


# Get expectation values at all times for the static MPF with approximate coeffs
approx_mpf_curve, approx_mpf_curve_error = [], []
for trotter_expvals, trotter_stds in zip(
    mpf_expvals_all_times, mpf_stds_all_times
):
    mpf_std = np.sqrt(
        sum(
            [
                (coeff**2) * (std**2)
                for coeff, std in zip(coeffs_approx.value, trotter_stds)
            ]
        )
    )
    approx_mpf_curve_error.append(mpf_std)
    approx_mpf_curve.append(trotter_expvals @ coeffs_approx.value)

plt.errorbar(
    trotter_times,
    approx_mpf_curve,
    yerr=approx_mpf_curve_error,
    markersize=4,
    marker="o",
    label="Static MPF - Approx",
    color="orange",
)


# Get expectation values at all times for the dynamic MPF
dynamic_mpf_curve, dynamic_mpf_curve_error = [], []
for trotter_expvals, trotter_stds, dynamic_coeffs in zip(
    mpf_expvals_all_times, mpf_stds_all_times, mpf_dynamic_coeffs_list
):
    mpf_std = np.sqrt(
        sum(
            [
                (coeff**2) * (std**2)
                for coeff, std in zip(dynamic_coeffs, trotter_stds)
            ]
        )
    )
    dynamic_mpf_curve_error.append(mpf_std)
    dynamic_mpf_curve.append(trotter_expvals @ dynamic_coeffs)

plt.errorbar(
    trotter_times,
    dynamic_mpf_curve,
    yerr=dynamic_mpf_curve_error,
    markersize=4,
    marker="o",
    label="Dynamic MPF",
    color="pink",
)


# Exact expectation values
plt.plot(
    exact_evolution_times,
    exact_expvals,
    color="red",
    linestyle="--",
    label="Exact time-evolution",
)

plt.title(f"$\\langle Z_{{{L//2-1}}} Z_{{{L//2}}} \\rangle$ vs time")
plt.xlabel("Time")
plt.ylabel("Expectation Value")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
plt.grid(alpha=0.1)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/multi-product-formula/extracted-outputs/35042576-0.avif" alt="Output of the previous code cell" />

##### $x$ mit einem Näherungsmodell optimieren
Es kann vorkommen, dass die L1-Norm für den gewählten Satz von $k_j$-Werten als zu hoch eingestuft wird.
Wenn das der Fall ist und du keinen anderen Satz von $k_j$-Werten wählen kannst, kannst du statt einer exakten Lösung eine Näherungslösung für das LGS verwenden.

Verwende dazu einfach [setup_approximate_model](https://qiskit.github.io/qiskit-addon-mpf/stubs/qiskit_addon_mpf.static.setup_approximate_model.html), um eine andere [cvxpy.Problem](https://www.cvxpy.org/api_reference/cvxpy.problems.html#cvxpy.Problem)-Instanz zu konstruieren, die die L1-Norm auf einen gewählten Schwellenwert beschränkt und dabei die Differenz zwischen $Ax$ und $b$ minimiert.

In [22]:
# -------------------------Step 1-------------------------
L = 50
coupling_map = CouplingMap.from_line(L, bidirectional=False)

# XXZ Hamiltonian with random couplings (Ref. [3])
np.random.seed(0)
even_edges = list(coupling_map.get_edges())[::2]
odd_edges = list(coupling_map.get_edges())[1::2]

Js = np.random.uniform(0.5, 1.5, size=L)
hamiltonian = SparsePauliOp(Pauli("I" * L))
for i, edge in enumerate(even_edges + odd_edges):
    hamiltonian += SparsePauliOp.from_sparse_list(
        [
            ("XX", (edge), 2 * Js[i]),
            ("YY", (edge), 2 * Js[i]),
            ("ZZ", (edge), 4 * Js[i]),
        ],
        num_qubits=L,
    )

observable = SparsePauliOp.from_sparse_list(
    [("ZZ", (L // 2 - 1, L // 2), 1.0)], num_qubits=L
)

total_time = 3
mpf_trotter_steps = [3, 4, 6]
order = 2
symmetric = True

# Static coefficients
lse = setup_static_lse(mpf_trotter_steps, order=order, symmetric=symmetric)
mpf_coeffs = lse.solve()
print(f"Static coefficients: {mpf_coeffs}")
print(f"L1 norm: {np.linalg.norm(mpf_coeffs, ord=1)}")

model_approx, coeffs_approx = setup_sum_of_squares_problem(
    lse, max_l1_norm=2.0
)
model_approx.solve()
print(f"Approximate coefficients: {coeffs_approx.value}")
print(f"L1 norm (approx): {np.linalg.norm(coeffs_approx.value, ord=1)}")

# -------------------------Dynamic coefficients-------------------------
single_2nd_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=order)
)
single_2nd_order_circ = pm.run(single_2nd_order_circ)

layers = slice_by_depth(single_2nd_order_circ, max_slice_depth=1)
models = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz") for layer in layers
]

approx_factory = partial(
    LayerwiseEvolver,
    layers=models,
    options={
        "preserve_norm": False,
        "trunc_params": {"chi_max": 64, "svd_min": 1e-8, "trunc_cut": None},
        "max_delta_t": 4,
    },
)

single_4th_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=4)
)
single_4th_order_circ = pm.run(single_4th_order_circ)
exact_model_layers = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz")
    for layer in slice_by_depth(single_4th_order_circ, max_slice_depth=1)
]

exact_factory = partial(
    LayerwiseEvolver,
    layers=exact_model_layers,
    dt=0.1,
    options={
        "preserve_norm": False,
        "trunc_params": {"chi_max": 64, "svd_min": 1e-8, "trunc_cut": None},
        "max_delta_t": 3,
    },
)


def identity_factory():
    return MPOState.initialize_from_lattice(models[0].lat, conserve=True)


mps_initial_state = MPS_neel_state(models[0].lat)

print(f"Computing dynamic coefficients for time={total_time}")
lse_dyn = setup_dynamic_lse(
    mpf_trotter_steps,
    total_time,
    identity_factory,
    exact_factory,
    approx_factory,
    mps_initial_state,
)
problem, coeffs_dyn = setup_frobenius_problem(lse_dyn)
try:
    problem.solve()
    mpf_dynamic_coeffs = coeffs_dyn.value
except Exception as error:
    mpf_dynamic_coeffs = np.zeros(len(mpf_trotter_steps))
    print(error, "Calculation Failed")

# -------------------------Step 1 (cont): Build circuits-------------------------
mpf_circuits = []
for k in mpf_trotter_steps:
    circuit = QuantumCircuit(L)
    circuit.x([i for i in range(L) if i % 2])
    trotter_circ = generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=k, order=order),
        time=total_time,
    )
    circuit.compose(trotter_circ, qubits=range(L), inplace=True)
    mpf_circuits.append(circuit)

# Baseline "single deep circuit" comparison run with k=10 Trotter steps.
# Its two-qubit depth is deeper than the deepest MPF constituent (k_max=6) plus
# the overhead of running multiple circuits, pushing it into the noise-limited
# regime where MPF is expected to outperform. It does NOT target the MPF's effective
# Trotter error (which would require many more steps).
comp_circuit = QuantumCircuit(L)
comp_circuit.x([i for i in range(L) if i % 2])
trotter_circ = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=10, order=order),
    time=total_time,
)
comp_circuit.compose(trotter_circ, qubits=range(L), inplace=True)
mpf_circuits.append(comp_circuit)

Static coefficients: [ 0.42857143 -1.82857143  2.4       ]
L1 norm: 4.65714285714286
Approximate coefficients: [-0.4942491   0.40206845  1.09218065]
L1 norm (approx): 1.9884981979026675
Computing dynamic coefficients for time=3


We now optimize the circuits for the chosen backend. We use the Qiskit preset pass manager at `optimization_level=3`, which automatically selects a good set of physical qubits and routes each circuit onto the device topology.

In [23]:
# -------------------------Step 2-------------------------
service = QiskitRuntimeService()
# backend = service.least_busy(operational=True, simulator=False, min_num_qubits=L)
backend = service.backend("ibm_fez")
print(backend)

transpiler = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)
transpiled_circuits = [transpiler.run(circ) for circ in mpf_circuits]

isa_observables = [
    observable.apply_layout(circ.layout) for circ in transpiled_circuits
]

<IBMBackend('ibm_fez')>


Beachte, dass du vollständige Freiheit darüber hast, wie du dieses Optimierungsproblem löst – du kannst den Optimierungs-Solver, seine Konvergenzschwellen und weiteres anpassen.
Lies dazu die entsprechende Anleitung [How to use the approximate model](https://qiskit.github.io/qiskit-addon-mpf/how_tos/using_approximate_model.html).
#### Dynamische MPF-Koeffizienten
Im vorherigen Abschnitt haben wir eine statische MPF eingeführt, die die Standard-Trotter-Näherung verbessert. Diese statische Variante minimiert den Approximationsfehler jedoch nicht notwendigerweise. Konkret ist die statische MPF $\mu^S(t)$ nicht die optimale Projektion von $\rho(t)$ auf den Unterraum, der von den Produkt-Formel-Zuständen ${\rho_{k_i}(t)}_{i=1}^r$ aufgespannt wird.

Um diesem Problem zu begegnen, betrachten wir eine dynamische MPF (eingeführt in Ref. [\[2\]](#references) und experimentell demonstriert in Ref. [\[3\]](#references)), die den Approximationsfehler in der Frobenius-Norm tatsächlich minimiert. Formal minimieren wir:

$$
\|\rho(t) - \mu^D(t)\|_F^2 \;=\; \mathrm{Tr}\bigl[ \left( \rho(t) - \mu^D(t)\right)^2 \bigr],
$$

bezüglich einiger Koeffizienten $x_i(t)$ zu jedem Zeitpunkt $t$. Der *optimale* Projektor in der Frobenius-Norm ist dann $\mu^D(t) \;=\; \sum_{i=1}^r x_i(t)\,\rho_{k_i}(t)$, und wir nennen $\mu^D(t)$ die *dynamische* MPF. Setzt man die obigen Definitionen ein, ergibt sich:

$$
\|\rho(t) - \mu^D(t)\|_F^2
\;=\; \\
= \mathrm{Tr}\bigl[ \left( \rho(t) - \mu^D(t)\right)^2 \bigr]
\;=\; \\
= \mathrm{Tr}\bigl[ \left( \rho(t) - \sum_{i=1}^r x_i(t)\,\rho_{k_i}(t) \right) \left(  \rho(t) - \sum_{j=1}^r x_j(t)\,\rho_{k_j}(t) \right) \bigr]
\;=\; \\
= 1 \;+\; \sum_{i,j=1}^r M_{i,j}(t)\,x_i(t)\,x_j(t)
\;-\;
2 \sum_{i=1}^r L_i^{\mathrm{exact}}(t)\,x_i(t),
$$

wobei $M_{i,j}(t)$ die *Gram-Matrix* ist, definiert durch:

$$
M_{i,j}(t) \;=\; \mathrm{Tr}\bigl[\rho_{k_i}(t)\,\rho_{k_j}(t)\bigr]
\;=\;
\bigl|\langle \psi_{\mathrm{in}} \!\mid S\bigl(t/k_i\bigr)^{-k_i}\,S\bigl(t/k_j\bigr)^{k_j} \!\mid \psi_{\mathrm{in}} \rangle \bigr|^2.
$$

und

$$
L_i^{\mathrm{exact}}(t) = \mathrm{Tr}[\rho(t)\,\rho_{k_i}(t)]
$$

die Überlappung zwischen dem exakten Zustand $\rho(t)$ und jeder Produkt-Formel-Näherung $\rho_{k_i}(t)$ darstellt. In der Praxis können diese Überlappungen aufgrund von Rauschen oder eingeschränktem Zugang zu $\rho(t)$ nur näherungsweise gemessen werden.

Hier ist $\lvert\psi_{\mathrm{in}}\rangle$ der Anfangszustand und $S(\cdot)$ die in der Produkt-Formel angewendete Operation. Indem man die Koeffizienten $x_i(t)$ wählt, die diesen Ausdruck minimieren (und dabei näherungsweise Überlappungsdaten berücksichtigt, wenn $\rho(t)$ nicht vollständig bekannt ist), erhält man die „beste" (im Sinne der Frobenius-Norm) dynamische Näherung von $\rho(t)$ im MPF-Unterraum. Die Größen $L_i(t)$ und $M_{i,j}(t)$ lassen sich effizient mit Tensornetzwerk-Methoden berechnen [\[3\]](#references). Das MPF-Qiskit-Addon bietet mehrere „Backends" für diese Berechnung. Das folgende Beispiel zeigt die flexibelste Vorgehensweise; die Dokumentation des [TeNPy-lagerbasierten Backends](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.backends.tenpy_layers.html#module-qiskit_addon_mpf.backends.tenpy_layers) erläutert dies ebenfalls ausführlich. Für diese Methode startest du vom Schaltkreis, der die gewünschte Zeitentwicklung implementiert, und erstellst Modelle, die diese Operationen anhand der Schichten des entsprechenden Schaltkreises darstellen. Schließlich wird ein `Evolver`-Objekt erzeugt, mit dem die zeitentwickelten Größen $M_{i,j}(t)$ und $L_i(t)$ berechnet werden können. Wir beginnen mit der Erstellung des `Evolver`-Objekts für die näherungsweise Zeitentwicklung ([`ApproxEvolverFactory`](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.dynamic.html#qiskit_addon_mpf.dynamic.ApproxEvolverFactory)), die durch die Schaltkreise implementiert wird. Achte dabei besonders auf die Variable `order`, damit die Werte übereinstimmen. Beachte, dass bei der Erzeugung der Schaltkreise für die näherungsweise Zeitentwicklung Platzhalterwerte für `time = 1.0` und die Anzahl der Trotter-Schritte (`reps=1`) verwendet werden. Die korrekten Näherungsschaltkreise werden dann vom dynamischen Problemlöser in `setup_dynamic_lse` erzeugt.

In [24]:
# -------------------------Step 3-------------------------
estimator = Estimator(mode=backend)
estimator.options.default_shots = 30000

# Error suppression/mitigation
estimator.options.dynamical_decoupling.enable = True
estimator.options.twirling.enable_gates = True
estimator.options.twirling.enable_measure = True
estimator.options.twirling.num_randomizations = "auto"
estimator.options.twirling.strategy = "active-accum"
estimator.options.resilience.measure_mitigation = True
estimator.options.experimental.execution_path = "gen3-turbo"

estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 1.2, 1.4)
estimator.options.resilience.zne.extrapolator = "linear"

estimator.options.environment.job_tags = ["TUT_MPF"]

job_50 = estimator.run(
    [
        (circ, observable)
        for circ, observable in zip(transpiled_circuits, isa_observables)
    ]
)

> **Warning:** Die Optionen von `LayerwiseEvolver`, die die Details der Tensornetzwerk-Simulation bestimmen, müssen sorgfältig gewählt werden, um ein schlecht definiertes Optimierungsproblem zu vermeiden.
Dann richten wir den exakten Evolver ein (zum Beispiel [`ExactEvolverFactory`](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.dynamic.html#qiskit_addon_mpf.dynamic.ExactEvolverFactory)), der ein [`Evolver`](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.backends.html#qiskit_addon_mpf.backends.Evolver)-Objekt zurückgibt, das die wahre oder „Referenz"-Zeitentwicklung berechnet. Realistischerweise würde man die exakte Entwicklung durch eine Suzuki–Trotter-Formel höherer Ordnung oder eine andere zuverlässige Methode mit kleinem Zeitschritt annähern. Im Folgenden nähern wir den exakt zeitentwickelten Zustand mit einer Suzuki-Trotter-Formel vierter Ordnung mit kleinem Zeitschritt `dt=0.1` an, was bedeutet, dass die Anzahl der Trotter-Schritte zur Zeit $t$ gleich $k=t/dt$ ist. Wir geben außerdem einige TeNPy-spezifische Trunkierungsoptionen an, um die maximale Bond-Dimension des zugrunde liegenden Tensornetzwerks sowie die minimalen Singulärwerte der aufgeteilten Tensornetzwerk-Bonds zu begrenzen. Diese Parameter können die Genauigkeit des mit den dynamischen MPF-Koeffizienten berechneten Erwartungswerts beeinflussen, daher ist es wichtig, einen Bereich von Werten zu erkunden, um die optimale Balance zwischen Rechenzeit und Genauigkeit zu finden. Beachte, dass die Berechnung der MPF-Koeffizienten nicht auf dem Erwartungswert der PF aus der Hardware-Ausführung basiert und daher in der Nachverarbeitung angepasst werden kann.

In [25]:
# -------------------------Step 4-------------------------
result = job_50.result()
evs = [res.data.evs for res in result]
std = [res.data.stds for res in result]

print(evs)
print(std)

[array(-0.07916195), array(-0.04479681), array(-0.2560756), array(-0.06045848)]
[array(0.04605538), array(0.10056336), array(0.14426151), array(0.04059092)]


In [26]:
exact_mpf_std = np.sqrt(
    sum([(coeff**2) * (std**2) for coeff, std in zip(mpf_coeffs, std[:3])])
)
print(
    "Exact static MPF expectation value: ",
    evs[:3] @ mpf_coeffs,
    "+-",
    exact_mpf_std,
)
approx_mpf_std = np.sqrt(
    sum(
        [
            (coeff**2) * (std**2)
            for coeff, std in zip(coeffs_approx.value, std[:3])
        ]
    )
)
print(
    "Approximate static MPF expectation value: ",
    evs[:3] @ coeffs_approx.value,
    "+-",
    approx_mpf_std,
)
dynamic_mpf_std = np.sqrt(
    sum(
        [
            (coeff**2) * (std**2)
            for coeff, std in zip(mpf_dynamic_coeffs, std[:3])
        ]
    )
)
print(
    "Dynamic MPF expectation value: ",
    evs[:3] @ mpf_dynamic_coeffs,
    "+-",
    dynamic_mpf_std,
)

Exact static MPF expectation value:  -0.5665938395816946 +- 0.3925273058119915
Approximate static MPF expectation value:  -0.25856647611537903 +- 0.164249927266166
Dynamic MPF expectation value:  -0.12667812062949296 +- 0.06059471006973169


In [27]:
sym = {3: "^", 4: "s", 6: "p"}
for k, step in enumerate(mpf_trotter_steps):
    plt.errorbar(
        k,
        evs[k],
        yerr=std[k],
        alpha=0.5,
        markersize=4,
        marker=sym[step],
        color="grey",
        label=f"{mpf_trotter_steps[k]} Trotter steps",
    )

plt.errorbar(
    3,
    evs[-1],
    yerr=std[-1],
    alpha=0.5,
    markersize=8,
    marker="x",
    color="blue",
    label="10 Trotter steps",
)

plt.errorbar(
    4,
    evs[:3] @ mpf_coeffs,
    yerr=exact_mpf_std,
    markersize=4,
    marker="o",
    color="purple",
    label="Static MPF",
)

plt.errorbar(
    5,
    evs[:3] @ coeffs_approx.value,
    yerr=approx_mpf_std,
    markersize=4,
    marker="o",
    color="orange",
    label="Approximate static MPF",
)

plt.errorbar(
    6,
    evs[:3] @ mpf_dynamic_coeffs,
    yerr=dynamic_mpf_std,
    markersize=4,
    marker="o",
    color="pink",
    label="Dynamic MPF",
)

exact_obs = -0.24384471447172074  # Calculated via Tensor Network calculation
plt.axhline(
    y=exact_obs, linestyle="--", color="red", label="Exact time-evolution"
)

plt.title(
    f"$\\langle Z_{{{L//2-1}}} Z_{{{L//2}}} \\rangle$ at time {total_time} for the different methods"
)
plt.xlabel("Method")
plt.ylabel("Expectation Value")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
plt.grid(alpha=0.1)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/multi-product-formula/extracted-outputs/64360d85-0.avif" alt="Output of the previous code cell" />

A few observations about the hardware results above:

- **Going deeper is not free on hardware.** The single-circuit baselines tell the story directly: the $k = 6$ circuit is essentially exact ($-0.256$ versus the reference $-0.244$), yet the deeper $k = 10$ baseline is *worse* ($-0.061$, off by $\sim 0.18$), not better. Once the Trotter error is already small, adding steps mostly deepens the circuit and accumulates more gate noise and decoherence. This is precisely the regime MPFs are built for: reach the accuracy of a deep circuit using only shallow constituents.

- **A small-norm MPF beats the deep single circuit.** The approximate-static MPF (capped at $\|x\|_1 \approx 2$) lands at $-0.259$, within $\sim 0.015$ of the reference and far closer than the $k = 10$ baseline. The dynamic MPF ($-0.127$) also comfortably beats that baseline. Both combine only the shallow $k_j = [3, 4, 6]$ circuits, yet recover an answer the deep single circuit could not.

- **Coefficient norm matters more than mathematical optimality.** The exact-static MPF has $\|x\|_1 = 4.66$ and is the *worst* estimator of all ($-0.567$, off by more than $0.3$): the large coefficient norm amplifies the residual gate noise, decoherence, and ZNE error on each $\langle A \rangle_{k_j}$ by roughly the same factor, overwhelming the Trotter-error cancellation it buys. Capping the norm (the approximate-static solver, $\|x\|_1 \approx 2$) removes this overwhelm and gives the best estimate &mdash; even though its coefficients no longer cancel the leading Trotter error exactly.

- **Individual shallow circuits can still be competitive.** The lone $k = 6$ constituent ($-0.256$) is itself essentially exact here &mdash; on this run it is even marginally closer than the approximate-static MPF. The catch is that you do not know in advance *which* single $k$ sits in the sweet spot of "converged but not yet noise-limited," and the safe-looking choice of simply going deeper ($k = 10$) to guarantee Trotter convergence is exactly the one that fails. The MPF gives a principled combination of shallow circuits that does not require guessing the right depth.

The practical takeaway is that on hardware, MPFs should be paired with strong error mitigation on each individual $\langle A \rangle_{k_j}$, the coefficient $L_1$-norm should be kept modest (use the approximate solver, or the dynamic MPF), and the Trotter steps $k_j$ should be chosen so that $t/k_{\min} \lesssim 1$ &mdash; here $k_{\min} = 3$ at $t = 3$ gives $t/k_{\min} = 1$, keeping the constituents inside the convergent regime where the leading-error model the static MPF relies on is valid. With those choices, the small-norm MPFs here match a converged single circuit while the naive "just go deeper" baseline does not, recovering the depth-versus-accuracy advantage shown in Ref. [\[3\]](#references). Note also that individual runs are noisy &mdash; on a different submission of the same job (or a different backend), the exact ordering can shift; the robust trends are that small-$\|x\|_1$ MPFs do well, the large-$\|x\|_1$ exact-static MPF is amplified by hardware noise, and the over-deep single circuit is noise-limited.

## Next steps
<Admonition type="tip" title="Recommendations">

If you found this work interesting, you might be interested in the following material:
- [How to choose the Trotter steps for an MPF](https://qiskit.github.io/qiskit-addon-mpf/how_tos/choose_trotter_steps.html) — practical guidance on selecting $k_j$ values to avoid instabilities
- [How to use the approximate model](https://qiskit.github.io/qiskit-addon-mpf/how_tos/using_approximate_model.html) — tuning the $L_1$-norm constraint and solver options for the approximate static MPF
- [`qiskit-addon-mpf` API reference](https://qiskit.github.io/qiskit-addon-mpf/) — full documentation for static, dynamic, and backend modules
</Admonition>

## References

\[1] Vazquez, A. C., Egger, D. J., Ochsner, D., & Woerner, S. Well-conditioned multi-product formulas for hardware-friendly Hamiltonian simulation. [Quantum, 7, 1067 (2023)](https://quantum-journal.org/papers/q-2023-07-25-1067/)

\[2] Zhuk, S., Robertson, N. F., & Bravyi, S. Trotter error bounds and dynamic multi-product formulas for Hamiltonian simulation. [Physical Review Research, 6(3), 033309 (2024)](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.6.033309)

\[3] Robertson, N. F., et al. Tensor network enhanced dynamic multiproduct formulas. [arXiv:2407.17405 (2024)](https://arxiv.org/abs/2407.17405)